# NanoWM: watch a pretrained world model

Run the official **Point Maze NanoWM-B/2** checkpoint on recorded validation actions.
The output is an **open-loop prediction**, not an interactive game or a planning benchmark.

1. Open this notebook from the project repository in the Python environment configured for this checkout.
2. Make sure that environment has the project dependencies installed and can access a CUDA GPU.
3. Run all cells. **Left: recorded frames. Right: predicted frames.**

The first 3 frames are observed context; the remaining 17 are predictions. Background sharpness
alone does not establish motion accuracy: watch the green agent, especially later in each clip.
The notebook uses the Python interpreter running the current Jupyter kernel; it does not create a
separate environment or clone another repository. GPU, download speed, and sampling steps determine total time.

Model and VAE revisions, dataset checksum, source revision, seed, and clip identities are recorded.
The 718 MB dataset archive expands to about 30 GB in full; this demo extracts only the metadata
and the four required observation sequences. Allow at least 12 GB for assets and model files.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

REPO = Path.cwd().resolve()
if not (REPO / "scripts" / "prepare_pusht_demo.py").is_file() or not (REPO / "src" / "sample" / "rollout.py").is_file():
    raise RuntimeError("Open this notebook with the project repository as the working directory.")
DEMO_HOME = Path(os.environ.get("NANOWM_DEMO_HOME", str(REPO / "nanowm-pusht-demo"))).expanduser().resolve()
NUM_SAMPLES = 4
SAMPLING_STEPS = 50  # 15: quick preview; 50: default; 250: slower comparison, no quality guarantee
SEED = 42
BATCH_SIZE = 1  # keeps the memory footprint small
DEMO_HOME.mkdir(parents=True, exist_ok=True)
ASSETS = DEMO_HOME / "assets"
OUTPUT = DEMO_HOME / f"results-{SAMPLING_STEPS}-seed-{SEED}"

def run(argv, **kwargs):
    print("Running:", " ".join(map(str, argv)), flush=True)
    subprocess.run(list(map(str, argv)), check=True, **kwargs)

run([sys.executable, "-c", "import torch; assert torch.cuda.is_available(), 'CUDA GPU is not available in the current Python environment'; print(torch.cuda.get_device_name(0), torch.version.cuda)"])
print("Project:", REPO)
print("Workspace:", DEMO_HOME)


## 1. Prepare model and validation assets

The notebook runs the project code with the current Jupyter kernel interpreter and uses this checkout directly.
It does not install packages, create a virtual environment, or fetch a separate source revision. Re-running
cells keeps downloaded assets. Set `NANOWM_DEMO_HOME` before launching Jupyter to choose another workspace.


## 2. Download the official model and a reproducible validation preview

The OSF archive is verified with its published SHA-256. Only selected observation files are extracted;
all trajectory metadata remains available so the train/validation split and action normalization match
the checkpoint. The checkpoint and VAE use fixed Hugging Face revisions. Preparation is safe to rerun.

In [ ]:
#run([sys.executable,"-Xfrozen_modules=off","scripts/prepare_pusht_demo.py", "--root", str(ASSETS), "--num-samples", str(NUM_SAMPLES)], cwd=REPO)
from scripts.prepare_pusht_demo import prepare

prepare(Path(ASSETS),NUM_SAMPLES)

provenance = json.loads((ASSETS / "provenance.json").read_text())
print("Distinct validation episodes:", provenance["episodes"])


## 3. Generate predictions

Four distinct validation trajectories, in a fixed order, with the same random seed.
Increasing sampling steps costs time and does **not** guarantee a better trajectory for every clip.
Batch size affects random-number consumption; keep it unchanged when comparing runs.

In [ ]:
started = time.monotonic()
run([sys.executable, "src/sample/rollout.py",
     "--config", provenance["config"], "--ckpt", provenance["checkpoint"],
     "--save_path", str(OUTPUT), "--num_samples", str(NUM_SAMPLES), "--batch_size", str(BATCH_SIZE),
     "--rollout_length", "20", "--history_length", "3",
     "--num_sampling_steps", str(SAMPLING_STEPS), "--scheduling_mode", "sequential",
     "--history_stabilization_level", "0.02", "--fps", "8",
     "--seed", str(SEED), "--unique_trajectories"], cwd=REPO)
report = json.loads((OUTPUT / "report.json").read_text())
assert len(report["samples"]) == NUM_SAMPLES
assert [s["traj_idx"] for s in report["samples"]] == provenance["episodes"]
print(f"Finished in {time.monotonic()-started:.1f} seconds; peak model GPU allocation {report['peak_gpu_memory_gib']:.2f} GiB.")


## 4. Watch every result

**Left = recorded validation images. Right = model prediction.** The first 3 frames are context.
Play each video or scrub to the end; do not judge the moving agent from the mostly static background.
Metrics below cover only the 17 predicted frames, before video compression. The frozen-frame baseline
illustrates why high full-image PSNR/SSIM can be misleading in this small, mostly static scene.

In [ ]:
from IPython.display import display, Video, Markdown
for sample in report["samples"]:
    i = sample["sample_id"]
    display(Markdown(f"### Clip {i+1} · validation episode {sample['traj_idx']}\nRecorded (left) | predicted (right); 3 context frames + 17 predicted frames."))
    display(Video(filename=str(OUTPUT / f"sample_{i:04d}_compare.mp4"), embed=True, width=768, html_attributes='controls loop muted playsinline style="max-width:100%;height:auto"'))
rows = ["| Clip | Prediction PSNR / SSIM | Frozen last context PSNR / SSIM |", "|---|---|---|"]
for s in report["samples"]:
    a, b = s["prediction"], s["repeat_last_context"]
    rows.append(f"| {s['sample_id']+1} | {a['psnr_db']:.2f} dB / {a['ssim']:.4f} | {b['psnr_db']:.2f} dB / {b['ssim']:.4f} |")
display(Markdown("\n".join(rows)))

## Evaluate the checkpoint

In [ ]:
run([sys.executable, "src/sample/evaluate_metrics.py",
    "--video_dir", str(OUTPUT),
    "--history_length", "3" ,
    "--output_csv", "metrics.csv"], cwd=REPO)

## 5. Keep the videos

The result ZIP contains the videos, metrics, and reproducibility metadata;
it contains the videos, metrics, and reproducibility metadata. **File → Download → Download .ipynb**
also saves this notebook with its embedded video outputs. Viewing saved results needs no GPU.

In [ ]:
source_revision = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True, check=True).stdout.strip()
(OUTPUT / "provenance.json").write_text(json.dumps(dict(provenance, source_revision=source_revision), indent=2))
archive = shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT)
print("Saved result ZIP:", archive)


## Try another setting

Change `SAMPLING_STEPS` or `SEED` in the first code cell, then rerun that cell and steps 3–5.
Changing `NUM_SAMPLES` also requires step 2 to extract the additional observations.

This is an inference preview using prerecorded actions. It does not measure planning success,
prove a universal quality improvement from more sampling steps, or reproduce the paper's benchmark.
Native simulator environments are not required. Setup errors stop the notebook immediately; rerun the
failed cell after correcting a dependency, network, or GPU issue.